# Enterprise Track 0 · Preflight

**Goal:** find out what this workspace lets you do before you build anything, and get a precise list of what to ask an admin for.

The session notebooks run on Databricks Free Edition. This track runs on the agency workspace and uses what Free Edition doesn't have: your own Unity Catalog schema, Lakeflow pipelines with expectations, scheduled jobs with PagerDuty alerts, column masks and row filters, data-quality monitoring and system tables. Every notebook prints the SQL it runs, so you finish with SQL you can reuse.

**Before you start:** set `ENTERPRISE_CATALOG` in `lnl/config.py` to a catalog where you can create a schema. If you don't know one, run this notebook anyway: it lists the catalogs you can see.

> ⚠️ **Mock data only.** This track runs on the agency workspace, which is approved for criminal justice information, but every table here holds the course's fictional records. Real tables come later, with the data owner's approval (see `enterprise/README.md`).

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and checks your settings.
STEP = 0

import importlib
import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import config, enterprise as ent
importlib.reload(config)                                         # picks up edits to lnl/config.py without a restart
N = ent.prepare(STEP, "Preflight")

## 1. Unity Catalog and compute
Unity Catalog is the governance layer: every table, view, volume and function has a three-part name (`catalog.schema.object`), an owner, and grants. If the first check fails, this workspace (or this compute) isn't attached to Unity Catalog, and nothing else in the track will work.

In [ ]:
import os
results = {}
try:
    metastore = spark.sql("SELECT current_metastore()").first()[0]
    results["uc"] = ent.check("Unity Catalog", True, metastore)
except Exception as exc:
    results["uc"] = ent.check("Unity Catalog", False, f"not available on this compute ({type(exc).__name__})")

runtime = os.environ.get("DATABRICKS_RUNTIME_VERSION")
results["compute"] = ent.check("Compute", True, f"runtime {runtime}" if runtime else "serverless")
print(f"   Spark {spark.version}; ENTERPRISE_SERVERLESS = {config.ENTERPRISE_SERVERLESS} in lnl/config.py")

## 2. Where you can build
The track creates one schema for you (`ENTERPRISE_SCHEMA`, named after you) inside `ENTERPRISE_CATALOG`, plus a volume for incoming files. Creating them needs `USE CATALOG` and `CREATE SCHEMA` on the catalog. These two statements are the first real objects you create; they're safe to run more than once.

In [ ]:
display(spark.sql("SHOW CATALOGS"))

In [ ]:
if not config.ENTERPRISE_CATALOG:
    results["schema"] = ent.check("ENTERPRISE_CATALOG", False, "not set in lnl/config.py; pick one from the list above")
else:
    N = ent.names()
    try:
        ent.sql(f"CREATE SCHEMA IF NOT EXISTS {N['fq']} COMMENT 'Lunch & Learn enterprise track (mock data only)'")
        results["schema"] = ent.check("Your schema", True, N["fq"])
    except Exception as exc:
        results["schema"] = ent.check("Your schema", False, f"can't create {N['fq']}: {str(exc).splitlines()[0][:160]}")
    if results["schema"]:
        try:
            ent.sql(f"CREATE VOLUME IF NOT EXISTS {N['volume_name']} COMMENT 'Landing zone: source files arrive here'")
            results["volume"] = ent.check("Your landing volume", True, N["volume"])
        except Exception as exc:
            results["volume"] = ent.check("Your landing volume", False, str(exc).splitlines()[0][:160])

## 3. Pipelines, jobs and the Databricks SDK
Notebooks 02 and 04 create a Lakeflow pipeline and a job from Python with the Databricks SDK, the same API the UI and the CLI use, so nothing needs a terminal. Serverless environment versions ship different SDK versions (environment 4 has 0.49, environment 6 has 0.122); the track works on any of them, and uses the newer data-quality monitoring API in notebook 05 when the SDK is 0.68 or newer.

In [ ]:
try:
    import databricks.sdk
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    version = databricks.sdk.version.__version__
    results["sdk"] = ent.check("Databricks SDK", True, f"version {version}, signed in as {w.current_user.me().user_name}")
    results["dq_api"] = ent.check("Data-quality monitoring API", hasattr(w, "data_quality"),
                                  "" if hasattr(w, "data_quality") else
                                  "SDK older than 0.68: notebook 05 falls back to the older API. For the new one, "
                                  "set this notebook's environment version to 6 or newer in the Environment panel")
except Exception as exc:
    results["sdk"] = ent.check("Databricks SDK", False, str(exc).splitlines()[0][:160])

try:
    list(w.pipelines.list_pipelines(max_results=1))
    results["pipelines"] = ent.check("Lakeflow pipelines", True, "you can list pipelines")
except Exception as exc:
    results["pipelines"] = ent.check("Lakeflow pipelines", False, str(exc).splitlines()[0][:160])

## 4. Alerts
Jobs send failure alerts to **notification destinations**: email, Slack, Microsoft Teams, PagerDuty, or a webhook. Only a workspace admin can create them; once created, anyone can attach them to a job. PagerDuty uses an Events API v2 integration key. ServiceNow and Vivantio would be webhook destinations, with something on their side that turns the webhook's JSON into a ticket. Copy the names you want into `ALERT_DESTINATIONS` in `lnl/config.py`.

In [ ]:
try:
    destinations = [(d.display_name, d.destination_type.value if d.destination_type else "") for d in w.notification_destinations.list()]
    results["destinations"] = ent.check("Notification destinations", bool(destinations),
                                        f"{len(destinations)} available" if destinations else "none yet; job failures will only email you")
    for name, kind in destinations:
        print(f"     {kind:12} {name}{'   <- configured' if name in config.ALERT_DESTINATIONS else ''}")
except Exception as exc:
    results["destinations"] = ent.check("Notification destinations", False, str(exc).splitlines()[0][:160])

## 5. Groups and system tables
Column masks and row filters decide what to show by **account group** membership. Set `RESTRICTED_GROUP` to the group allowed to see race and religion, and `ANALYST_GROUP` to the group that gets the published view.

**System tables** (`system.access`, `system.lakeflow`, `system.data_quality_monitoring`) are the workspace's own records: lineage, job runs, audit events, quality results. An account admin enables them and grants access.

In [ ]:
for label, group in [("RESTRICTED_GROUP", config.RESTRICTED_GROUP), ("ANALYST_GROUP", config.ANALYST_GROUP)]:
    if not group:
        ent.check(label, False, "not set; masks will hide race and religion from everyone, which is safe")
        continue
    member = spark.sql(f"SELECT is_account_group_member('{group}')").first()[0]
    ent.check(label, True, f"{group} (you are {'a member' if member else 'not a member'})")

for table in ["system.access.table_lineage", "system.lakeflow.job_run_timeline", "system.access.audit",
              "system.data_quality_monitoring.table_results"]:
    try:
        spark.sql(f"SELECT 1 FROM {table} LIMIT 1").collect()
        results[table] = ent.check(table, True, "readable")
    except Exception:
        results[table] = ent.check(table, False, "no access (ask an admin to enable the schema and grant SELECT)")

## What to ask for
Everything marked ⚠️ above, in one list you can paste into a request. Serverless compute for notebooks, jobs and pipelines is the simplest setup; if it isn't enabled, set `ENTERPRISE_SERVERLESS = False` and fill in the classic-compute settings in `lnl/config.py`.

In [ ]:
asks = {
    "uc": "Attach this compute to Unity Catalog (use serverless or a UC-enabled access mode).",
    "schema": f"USE CATALOG and CREATE SCHEMA on catalog {config.ENTERPRISE_CATALOG or '<a sandbox catalog>'}.",
    "volume": "CREATE VOLUME on my schema (owners have it by default).",
    "sdk": "The databricks-sdk Python package on this compute.",
    "pipelines": "Permission to create Lakeflow pipelines (serverless preferred).",
    "destinations": "Notification destinations for job alerts: PagerDuty (Events API v2 key) and/or a webhook for ServiceNow.",
    "system.access.table_lineage": "SELECT on system.access (lineage).",
    "system.lakeflow.job_run_timeline": "SELECT on system.lakeflow (job and pipeline history).",
    "system.access.audit": "SELECT on system.access.audit (optional: who read what).",
    "system.data_quality_monitoring.table_results": "Anomaly detection results (optional).",
}
missing = [asks[k] for k, ok in results.items() if not ok and k in asks]
print("Nothing to ask for. Go to 01_unity_catalog." if not missing else "Ask your workspace admin for:\n- " + "\n- ".join(missing))